In [1]:
# 1. Config

import pandas as pd
from pathlib import Path
from collections import defaultdict

# ── Configure paths ───────────────────────────────────────────────────────────
BASE_DIR   = Path.home() / "Desktop" / "cri_runs" # parent of Anthropic/, Gemini/, OpenAI/
OUTPUT_DIR = Path("/Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/Project 1/Runs") # merged CSVs written here
# ─────────────────────────────────────────────────────────────────────────────

KEEP_COLS = ["TokenID", "FullText", "Target", "annotation", "confidence", "explanation"]

In [2]:
# 2. Discover

def discover_runs(base_dir: Path) -> dict:
    """
    Returns: {model_dir_name: {perm: {feature_name: csv_path}}}
    For each (model, feature, perm), selects the CSV inside the
    latest run_id folder (max by timestamp-prefixed folder name).
    """
    raw = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

    for model_dir in sorted(base_dir.iterdir()):
        if not model_dir.is_dir() or model_dir.name == "merged":
            continue
        for feature_dir in sorted(model_dir.iterdir()):
            if not feature_dir.is_dir():
                continue
            for run_id_dir in sorted(feature_dir.iterdir()):
                if not run_id_dir.is_dir():
                    continue
                for csv_path in run_id_dir.glob("*.csv"):
                    parts = csv_path.stem.split("_")
                    if len(parts) < 3:
                        continue
                    perm = parts[2]          # e.g. full, abl1, abl2, all1, …
                    raw[model_dir.name][feature_dir.name][perm].append(
                        (run_id_dir.name, csv_path)
                    )

    result = defaultdict(lambda: defaultdict(dict))
    for model, features in raw.items():
        for feature, perms in features.items():
            for perm, candidates in perms.items():
                latest_csv = max(candidates, key=lambda x: x[0])[1]
                result[model][perm][feature] = latest_csv

    return result


runs = discover_runs(BASE_DIR)
print(f"Found {len(runs)} model directory/ies: {sorted(runs)}")

Found 1 model directory/ies: ['Anthropic']


In [3]:
# 3. Validate

all_features = sorted({f for perms in runs.values() for feats in perms.values() for f in feats})
all_perms    = sorted({p for perms in runs.values() for p in perms})

print(f"Features ({len(all_features)}): {all_features}")
print(f"Permutations ({len(all_perms)}): {all_perms}\n")

any_missing = False
for model in sorted(runs):
    print(f"── {model} ──")
    for perm in all_perms:
        found   = set(runs[model].get(perm, {}).keys())
        missing = set(all_features) - found
        flag    = f"  ⚠ MISSING: {sorted(missing)}" if missing else ""
        print(f"  {perm:<10} {len(found):>2} features{flag}")
        if missing:
            any_missing = True
    print()

if not any_missing:
    print("✓ No missing features — all (model, perm) combinations are complete.")

Features (16): ['ASTrns', 'ASUncc', 'ASUnrg', 'AktAcc', 'AktAch', 'AktAct', 'AktState', 'ClCntfcl', 'ClNegPol', 'DtExp', 'DtRes', 'DtUniv', 'TLAspPfv', 'TLMdIrr', 'TLTS', 'VcPass']
Permutations (1): ['full']

── Anthropic ──
  full       16 features

✓ No missing features — all (model, perm) combinations are complete.


In [4]:
# 4. Merge & write

def merge_perm(feature_csv_map: dict) -> pd.DataFrame:
    dfs = []
    for feature, csv_path in sorted(feature_csv_map.items()):
        df = pd.read_csv(csv_path, usecols=KEEP_COLS)
        df = df.rename(columns={
            "annotation":  feature,
            "confidence":  f"{feature}_conf",
            "explanation": f"{feature}_explanation",
        })
        dfs.append(df)

    if not dfs:
        return pd.DataFrame()

    merged = dfs[0]
    for df in dfs[1:]:
        right = df.drop(columns=["FullText", "Target"], errors="ignore")
        merged = merged.merge(right, on="TokenID", how="outer")

    base_cols = ["TokenID", "FullText", "Target"]
    feat_cols = [c for c in merged.columns if c not in base_cols]
    return merged[base_cols + feat_cols]


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

written = []
for model in sorted(runs):
    model_out = OUTPUT_DIR / model
    model_out.mkdir(exist_ok=True)
    for perm in sorted(runs[model]):
        feature_csv_map = runs[model][perm]
        merged = merge_perm(feature_csv_map)
        out_path = model_out / f"{model}_{perm}.csv"
        merged.to_csv(out_path, index=False)
        written.append(out_path)
        print(f"✓ {model}/{perm}.csv  "
              f"({len(feature_csv_map)} features, {len(merged):,} tokens, "
              f"{merged.shape[1]} columns)")

print(f"\nDone — {len(written)} file(s) written to {OUTPUT_DIR}")

✓ Anthropic/full.csv  (16 features, 347 tokens, 51 columns)

Done — 1 file(s) written to /Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/Project 1/Runs
